# Bengaluru Climate Risk Scanner
## Notebook 02: Flood susceptibility model (Phase 2)

This notebook turns the Phase 1 layers into a **flood risk index (FRI)** for every 30 m cell of Bengaluru, then into **Low / Medium / High flood zones** that GeoServer will publish and the web app will query.

It is a **weighted overlay**, the standard multi-criteria method in GIS. Six factors are each scored from 1 (safe) to 5 (very risky), then combined with weights:

| Factor | Why it matters | Score 5 means |
|---|---|---|
| Height above nearest drainage (HAND) | Water collects in low ground next to streams | Less than 1.5 m above the nearest stream, drain or lake |
| Slope | Flat land drains slowly | Under 1° |
| Distance to lakes | Bengaluru floods start at overflowing lakes | Within 250 m |
| Distance to drains | Overflowing stormwater drains spread water | Within 100 m |
| Topographic wetness index (TWI) | Shows where runoff naturally gathers | Wettest 20% of the city |
| Built-up share | Concrete stops rain soaking in | 80% or more of the cell built-up |

FRI = 0.25 × HAND + 0.10 × slope + 0.20 × lakes + 0.20 × drains + 0.15 × TWI + 0.10 × built-up

The weights come from `config.json` (Notebook 01). An optional AHP cell shows how to justify them.

**Outputs** (all in `MyDrive/bengaluru-climate-risk/data/processed/`):

| File | What it holds | Used by |
|---|---|---|
| `flood_risk.tif` | FRI, 1 to 5, inside the city | GeoServer WMS (colour ramp) |
| `flood_class.tif` | 1 Low, 2 Medium, 3 High, 0 permanent water | Validation (Notebook 05) |
| `flood_factor_scores.tif` | The six 1–5 scores, one band each | Popups ("why is this High?") |
| `flood_factor_values.tif` | The six raw values (metres, degrees, index, share) | Popups ("320 m from the nearest lake") |
| `bengaluru.gpkg` → `flood_zones` | Low/Medium/High polygons | GeoServer WFS (property scoring) |
| `bengaluru.gpkg` → `ward_flood_summary` | Share of each ward in High risk | Ward view on the map |

**Before you start:** run Notebook 01 first. This notebook needs no Earth Engine login; everything runs in Colab from the files in Drive.

## 1. Install libraries
`pyflwdir` computes how water flows over the terrain (flow direction, upstream area, height above drainage).

In [ ]:
!pip install -q rasterio pyflwdir

## 2. Connect Google Drive and load the project configuration
Every setting comes from the `config.json` written by Notebook 01, so both notebooks always agree on paths, grid and weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil

PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
with open(f'{PROJECT_DIR}/config.json') as f:
    CONFIG = json.load(f)

RAW = CONFIG['paths']['raw']
OUT = CONFIG['paths']['processed']
GPKG = CONFIG['paths']['gpkg']
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)

print('City:', CONFIG['city'])
print('Elevation source:', CONFIG.get('dem_source_used', CONFIG['dem_source']))
print('Weights:', CONFIG['flood_weights'])

## 3. Model settings
The score breaks for each factor. A value is compared with the breaks from low to high; for example with slope breaks `[1, 2, 4, 8]`, a slope under 1° scores 5, 1–2° scores 4, 2–4° scores 3, 4–8° scores 2 and over 8° scores 1.

`class_quantiles` turns the FRI into classes: the riskiest 15% of city land is **High**, the next 30% **Medium**, the rest **Low**. So the classes are *relative* to Bengaluru. Notebook 05 checks them against the 2022 flood and may adjust these numbers; every change is recorded in `config.json`.

In [ ]:
FLOOD_MODEL = {
    "hand_breaks_m":      [1.5, 3, 6, 10],      # height above drainage: < 1.5 m scores 5, > 10 m scores 1
    "slope_breaks_deg":   [1, 2, 4, 8],         # < 1 degree scores 5, > 8 degrees scores 1
    "lake_breaks_m":      [250, 500, 1000, 2000],
    "drain_breaks_m":     [100, 250, 500, 1000],
    "builtup_breaks":     [0.2, 0.4, 0.6, 0.8], # >= 80% built-up scores 5
    "twi_quantiles":      [0.2, 0.4, 0.6, 0.8], # TWI scored by fifths of the city
    "stream_threshold_km2": 0.5,                # upstream area at which a cell counts as a stream (for HAND)
    "class_quantiles":    [0.55, 0.85],         # FRI percentiles: Low < 55th <= Medium < 85th <= High
    "min_zone_cells":     10,                   # zone patches smaller than 10 cells (0.9 ha) merge into neighbours
}

FACTORS = ['hand', 'slope', 'dist_lake', 'dist_drain', 'twi', 'builtup']
WEIGHT_KEYS = {'hand': 'elevation', 'slope': 'slope', 'dist_lake': 'dist_lake',
               'dist_drain': 'dist_drain', 'twi': 'twi', 'builtup': 'builtup'}
WEIGHTS = {f: CONFIG['flood_weights'][WEIGHT_KEYS[f]] for f in FACTORS}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9, 'Weights must add up to 1'
print('Weights used:', WEIGHTS)

## 4. Load the Phase 1 layers
The elevation and built-up rasters share one 30 m grid (Notebook 01 checked this). The vector layers are converted to the same metre-based CRS so they can be burned onto that grid.

In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio import features

with rasterio.open(f'{RAW}/dem_30m.tif') as src:
    dem = src.read(1, masked=True).astype('float32')
    profile = src.profile
    transform, crs = src.transform, src.crs
    shape = (src.height, src.width)
    res = src.res[0]

with rasterio.open(f'{RAW}/builtup_30m.tif') as src:
    assert (src.height, src.width) == shape and src.transform.almost_equals(transform), \
        'Built-up raster is not on the DEM grid: rerun Notebook 01'
    builtup = src.read(1, masked=True).filled(0).astype('float32')

# Fill any empty DEM cells with the mean so flow and slope calculations don't break
dem_valid = ~np.ma.getmaskarray(dem) & np.isfinite(dem.filled(np.nan))
dem_f = np.where(dem_valid, dem.filled(np.nan), np.nanmean(dem.filled(np.nan))).astype('float32')
if (~dem_valid).any():
    print(f'Filled {(~dem_valid).sum():,} empty DEM cells')

layers = gpd.list_layers(GPKG)['name'].tolist()
boundary = gpd.read_file(GPKG, layer='bbmp_boundary').to_crs(crs)
lakes    = gpd.read_file(GPKG, layer='lakes').to_crs(crs)
drains   = gpd.read_file(GPKG, layer='drains').to_crs(crs)

print(f'Grid: {shape[1]} x {shape[0]} cells of {res:.0f} m, CRS {crs}')
print(f'Elevation {dem_f.min():.0f} to {dem_f.max():.0f} m; lakes {len(lakes):,}; drains {len(drains):,}')
print('GeoPackage layers:', layers)

## 5. Burn the vectors onto the grid
`rasterize` marks every cell a shape touches: one mask for the city, one for lakes, one for drains. `all_touched=True` makes thin drain lines continuous instead of dotted.

In [ ]:
def burn(gdf, all_touched=False):
    """True where any shape of gdf covers the cell."""
    if len(gdf) == 0:
        return np.zeros(shape, dtype=bool)
    return features.rasterize(((g, 1) for g in gdf.geometry if g is not None and not g.is_empty),
                              out_shape=shape, transform=transform, fill=0,
                              all_touched=all_touched, dtype='uint8').astype(bool)

city_mask  = burn(boundary)
lake_mask  = burn(lakes)
drain_mask = burn(drains, all_touched=True)
land = city_mask & ~lake_mask          # city cells that are not permanent water

cell_km2 = res * res / 1e6
print(f'City: {city_mask.sum() * cell_km2:,.0f} km²  |  lakes in city: {(city_mask & lake_mask).sum() * cell_km2:,.1f} km²  '
      f'|  drain cells: {drain_mask.sum():,}')

## 6. Terrain factors: slope, flow, HAND and TWI
- **Slope**: how steep each cell is, from the change in height to its neighbours.
- **Flow direction and upstream area**: `pyflwdir` first fills small pits in the DEM (so water can always run off), then works out where each cell drains and how much land drains *into* it (upstream area).
- **Streams**: cells whose upstream area exceeds `stream_threshold_km2`, plus the OSM drains and lakes. Together they are the drainage network.
- **HAND (height above nearest drainage)**: how many metres a cell sits above the stream it drains to. Low HAND = water reaches it first when streams overflow. This is the "relative elevation" factor.
- **TWI (topographic wetness index)** = ln(upstream area per metre of contour / tan(slope)). High where a lot of water arrives and the ground is flat.

In [ ]:
import pyflwdir

# Slope in degrees
dzdy, dzdx = np.gradient(dem_f, res)
slope_deg = np.degrees(np.arctan(np.hypot(dzdx, dzdy))).astype('float32')

# Flow directions (pits are filled internally), then upstream area
flw = pyflwdir.from_dem(dem_f, nodata=-9999.0, transform=transform, latlon=False, outlets='edge')
upa_cells = flw.upstream_area(unit='cell').astype('float64')
upa_km2 = upa_cells * cell_km2

# Drainage network = streams from the terrain + OSM drains + lakes
streams = (upa_km2 >= FLOOD_MODEL['stream_threshold_km2']) | drain_mask | lake_mask
hand = flw.hand(drain=streams, elevtn=dem_f).astype('float32')
hand = np.where(hand < 0, np.nan, hand)        # cells that drain off the grid without meeting a stream

# TWI: specific catchment area (m) / tan(slope); flat cells get a tiny slope to avoid dividing by zero
sca = upa_cells * res
tan_b = np.maximum(np.tan(np.radians(slope_deg)), 0.001)
twi = np.log(sca / tan_b).astype('float32')

print(f'Slope in city: median {np.median(slope_deg[land]):.1f}°, 95th percentile {np.percentile(slope_deg[land], 95):.1f}°')
print(f'HAND in city:  median {np.nanmedian(hand[land]):.1f} m, cells without a value: {np.isnan(hand[land]).sum():,}')
print(f'TWI in city:   {np.percentile(twi[land], 5):.1f} to {np.percentile(twi[land], 95):.1f} (5th to 95th percentile)')
print(f'Stream cells:  {streams.sum():,} ({(streams & city_mask).sum() * cell_km2:.1f} km² inside the city)')

## 7. Distance factors
`distance_transform_edt` gives every cell its straight-line distance to the nearest lake (or drain) cell, counted in cells; multiplying by 30 gives metres. Lakes and drains outside the city count too, which is why Notebook 01 downloaded a 5 km margin.

In [ ]:
from scipy import ndimage

dist_lake  = (ndimage.distance_transform_edt(~lake_mask)  * res).astype('float32')
dist_drain = (ndimage.distance_transform_edt(~drain_mask) * res).astype('float32')

print(f'Distance to lakes in city:  median {np.median(dist_lake[land]):.0f} m, max {dist_lake[land].max():.0f} m')
print(f'Distance to drains in city: median {np.median(dist_drain[land]):.0f} m, max {dist_drain[land].max():.0f} m')

## 8. Score every factor from 1 (safe) to 5 (very risky)
`np.digitize` says which interval between the breaks a value falls in (0 to 4). For factors where *low* values are risky (HAND, slope, distances), the score is 5 minus that interval; for built-up and TWI, where *high* values are risky, it is 1 plus the interval. HAND cells with no value (they drain off the map edge) get the middle score, 3.

In [ ]:
def score_low_is_risky(values, breaks):
    return (5 - np.digitize(values, breaks)).astype('uint8')

def score_high_is_risky(values, breaks):
    return (1 + np.digitize(values, breaks)).astype('uint8')

twi_breaks = np.quantile(twi[land], FLOOD_MODEL['twi_quantiles']).round(2).tolist()

scores = {
    'hand':       np.where(np.isnan(hand), 3, score_low_is_risky(np.nan_to_num(hand, nan=0), FLOOD_MODEL['hand_breaks_m'])).astype('uint8'),
    'slope':      score_low_is_risky(slope_deg, FLOOD_MODEL['slope_breaks_deg']),
    'dist_lake':  score_low_is_risky(dist_lake, FLOOD_MODEL['lake_breaks_m']),
    'dist_drain': score_low_is_risky(dist_drain, FLOOD_MODEL['drain_breaks_m']),
    'twi':        score_high_is_risky(twi, twi_breaks),
    'builtup':    score_high_is_risky(builtup, FLOOD_MODEL['builtup_breaks']),
}
values = {'hand': hand, 'slope': slope_deg, 'dist_lake': dist_lake,
          'dist_drain': dist_drain, 'twi': twi, 'builtup': builtup}

import pandas as pd
share = pd.DataFrame({f: [(scores[f][land] == s).mean() * 100 for s in range(1, 6)] for f in FACTORS},
                     index=[f'score {s}' for s in range(1, 6)]).round(1)
print('Share of city land with each score (%):')
print(share.to_string())
print('\nTWI breaks used:', twi_breaks)

## 9. Optional: justify the weights with AHP
The **Analytic Hierarchy Process** turns pairwise judgements ("HAND is twice as important as slope") into weights, and checks they are consistent. A consistency ratio (CR) below 0.10 is acceptable.

The matrix below is a starting judgement for Bengaluru's lake-and-drain flooding and gives weights close to the ones in `config.json`. Edit the judgements if you disagree, and set `USE_AHP_WEIGHTS = True` to use the AHP result instead of the config weights.

In [ ]:
USE_AHP_WEIGHTS = False

# Row factor compared with column factor: 3 = row moderately more important, 1/3 = less important
#              hand  slope lake  drain twi   builtup
AHP = np.array([[1,    3,    1,    1,    2,    3  ],   # hand
                [1/3,  1,    1/2,  1/2,  1/2,  1  ],   # slope
                [1,    2,    1,    1,    1,    2  ],   # dist_lake
                [1,    2,    1,    1,    1,    2  ],   # dist_drain
                [1/2,  2,    1,    1,    1,    2  ],   # twi
                [1/3,  1,    1/2,  1/2,  1/2,  1  ]])  # builtup

eigvals, eigvecs = np.linalg.eig(AHP)
k = np.argmax(eigvals.real)
ahp_w = np.abs(eigvecs[:, k].real); ahp_w /= ahp_w.sum()
n = len(AHP)
ci = (eigvals[k].real - n) / (n - 1)
cr = ci / 1.24                                  # 1.24 = random index for 6 factors
print(pd.DataFrame({'config weight': [WEIGHTS[f] for f in FACTORS], 'AHP weight': ahp_w.round(3)},
                   index=FACTORS).to_string())
print(f'\nConsistency ratio: {cr:.3f}', '(OK, below 0.10)' if cr < 0.10 else '(too inconsistent: revise the judgements)')

if USE_AHP_WEIGHTS:
    assert cr < 0.10, 'Fix the AHP judgements before using them'
    WEIGHTS = {f: round(float(w), 3) for f, w in zip(FACTORS, ahp_w)}
    print('Using AHP weights:', WEIGHTS)

## 10. Combine into the flood risk index and the three classes
- **FRI** = the weighted sum of the six scores, so it runs from 1 (all factors safe) to 5 (all very risky). It is calculated for city land only; permanent lakes are shown as water, not as risk.
- **Classes**: the FRI values at the 55th and 85th percentile of city land become the Low/Medium and Medium/High breaks.

In [ ]:
fri = sum(WEIGHTS[f] * scores[f].astype('float32') for f in FACTORS)
fri = np.where(land, fri, np.nan).astype('float32')

class_breaks = np.nanquantile(fri, FLOOD_MODEL['class_quantiles']).round(3).tolist()

NODATA_CLASS = 255
flood_class = np.full(shape, NODATA_CLASS, dtype='uint8')
flood_class[city_mask & lake_mask] = 0                                    # permanent water
flood_class[land] = (1 + np.digitize(fri[land], class_breaks)).astype('uint8')   # 1 Low, 2 Medium, 3 High

LEVELS = {1: 'Low', 2: 'Medium', 3: 'High'}
summary = pd.DataFrame({
    'area_km2': [((flood_class == c).sum() * cell_km2) for c in (1, 2, 3)],
}, index=[LEVELS[c] for c in (1, 2, 3)])
summary['share_%'] = (100 * summary['area_km2'] / summary['area_km2'].sum()).round(1)
summary['area_km2'] = summary['area_km2'].round(1)

print(f'FRI in city: {np.nanmin(fri):.2f} to {np.nanmax(fri):.2f}, mean {np.nanmean(fri):.2f}')
print(f'Class breaks (FRI): Low < {class_breaks[0]} <= Medium < {class_breaks[1]} <= High')
print(summary.to_string())

## 11. Look at the factors and the result
Check that the pattern makes sense: High zones should follow the valleys and lake chains (for example Bellandur–Varthur in the south-east and the Vrishabhavathi valley in the south-west), not ridges or random speckle.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

extent = [transform.c, transform.c + shape[1] * res, transform.f - shape[0] * res, transform.f]
titles = {'hand': 'Height above drainage', 'slope': 'Slope', 'dist_lake': 'Distance to lakes',
          'dist_drain': 'Distance to drains', 'twi': 'Wetness index (TWI)', 'builtup': 'Built-up share'}
score_cmap = ListedColormap(['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027'])
score_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], 5)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, f in zip(axes.flat, FACTORS):
    im = ax.imshow(np.where(city_mask, scores[f], np.nan), cmap=score_cmap, norm=score_norm, extent=extent)
    boundary.boundary.plot(ax=ax, color='black', linewidth=0.6)
    ax.set_title(f'{titles[f]}  (weight {WEIGHTS[f]:.2f})'); ax.set_axis_off()
cbar = fig.colorbar(im, ax=axes, shrink=0.6, ticks=[1, 2, 3, 4, 5])
cbar.set_label('Score: 1 safe → 5 very risky')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
im = axes[0].imshow(fri, cmap='RdYlGn_r', vmin=1, vmax=5, extent=extent)
fig.colorbar(im, ax=axes[0], shrink=0.7, label='Flood risk index (1–5)')
axes[0].set_title('Flood risk index')

class_cmap = ListedColormap(['#4a90d9', '#1a9850', '#fee08b', '#d73027'])
axes[1].imshow(np.where(flood_class == NODATA_CLASS, np.nan, flood_class), cmap=class_cmap,
               norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4), extent=extent, interpolation='nearest')
axes[1].set_title('Flood classes: water, Low, Medium, High')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=c, label=l) for c, l in
                        zip(class_cmap.colors, ['Permanent water', 'Low', 'Medium', 'High'])], loc='lower left')
for ax in axes:
    boundary.boundary.plot(ax=ax, color='black', linewidth=0.8); ax.set_axis_off()
plt.show()

## 12. Save the rasters
Written to fast local storage first, then copied to Drive. Band names are stored inside the multi-band files so GeoServer and QGIS show them.

In [ ]:
def save_tif(name, arrays, dtype, nodata, band_names=None):
    arrays = arrays if isinstance(arrays, list) else [arrays]
    prof = profile.copy()
    prof.update(driver='GTiff', dtype=dtype, count=len(arrays), nodata=nodata,
                compress='deflate', tiled=True, blockxsize=256, blockysize=256)
    local = f'{WORK}/{name}'
    with rasterio.open(local, 'w', **prof) as dst:
        for i, a in enumerate(arrays, start=1):
            dst.write(a.astype(dtype), i)
            if band_names:
                dst.set_band_description(i, band_names[i - 1])
    shutil.copy(local, f'{OUT}/{name}')
    print(f'Saved {OUT}/{name}  ({os.path.getsize(local) / 1e6:.1f} MB)')

save_tif('flood_risk.tif', np.where(np.isnan(fri), -9999, fri), 'float32', -9999)
save_tif('flood_class.tif', flood_class, 'uint8', NODATA_CLASS)
save_tif('flood_factor_scores.tif',
         [np.where(city_mask, scores[f], 0) for f in FACTORS], 'uint8', 0, band_names=FACTORS)
save_tif('flood_factor_values.tif',
         [np.where(city_mask & np.isfinite(values[f]), values[f], -9999) for f in FACTORS], 'float32', -9999,
         band_names=['hand_m', 'slope_deg', 'dist_lake_m', 'dist_drain_m', 'twi', 'builtup_share'])

## 13. Flood zones as polygons (for WFS)
The web app asks GeoServer "which flood zone contains this property?", which needs polygons, not a raster.

1. **Sieve**: patches smaller than `min_zone_cells` are merged into the surrounding class, which removes salt-and-pepper speckle and keeps the layer small.
2. **Polygonise**: each connected patch of one class becomes one polygon.
3. Each polygon gets `risk_class` (1–3), `risk_level` (Low/Medium/High), its area and its mean FRI, then is saved to the GeoPackage in latitude/longitude like the other layers.

In [ ]:
from shapely.geometry import shape as to_shape

zones_arr = np.where(land, flood_class, 0).astype('uint8')
sieved = features.sieve(zones_arr, size=FLOOD_MODEL['min_zone_cells'], connectivity=8)
sieved = np.where(land, sieved, 0).astype('uint8')
sieved[land & (sieved == 0)] = flood_class[land & (sieved == 0)]   # land cells merged into water: keep own class

polys, classes = [], []
for geom, val in features.shapes(sieved, mask=sieved > 0, transform=transform, connectivity=8):
    polys.append(to_shape(geom)); classes.append(int(val))

flood_zones = gpd.GeoDataFrame({'risk_class': classes}, geometry=polys, crs=crs)
flood_zones['risk_level'] = flood_zones['risk_class'].map(LEVELS)
flood_zones['area_m2'] = flood_zones.area.round(0)

# Mean FRI of each polygon, from the raster
zone_id = features.rasterize(((g, i + 1) for i, g in enumerate(flood_zones.geometry)),
                             out_shape=shape, transform=transform, fill=0, dtype='int32')
inside = (zone_id > 0) & land
sums = np.bincount(zone_id[inside], weights=fri[inside], minlength=len(flood_zones) + 1)
counts = np.bincount(zone_id[inside], minlength=len(flood_zones) + 1)
flood_zones['mean_fri'] = np.round(sums[1:] / np.maximum(counts[1:], 1), 2)

flood_zones = flood_zones.to_crs(CONFIG['web_crs'])
print(f'{len(flood_zones):,} polygons')
print(flood_zones.groupby('risk_level')['area_m2'].agg(['count', 'sum']).assign(
      km2=lambda d: (d['sum'] / 1e6).round(1)).drop(columns='sum').to_string())

## 14. Ward summary (optional)
If the ward layer downloaded in Notebook 01, this counts how much of each ward is in each flood class. The map's ward view will shade wards by `high_share_pct`.

In [ ]:
ward_summary = None
if 'wards' in layers:
    wards = gpd.read_file(GPKG, layer='wards').to_crs(crs).reset_index(drop=True)
    ward_id = features.rasterize(((g, i + 1) for i, g in enumerate(wards.geometry)),
                                 out_shape=shape, transform=transform, fill=0, dtype='int32')
    sel = (ward_id > 0) & land
    n = len(wards) + 1
    total = np.bincount(ward_id[sel], minlength=n)[1:]
    for c in (1, 2, 3):
        cnt = np.bincount(ward_id[sel], weights=(flood_class[sel] == c), minlength=n)[1:]
        wards[f'{LEVELS[c].lower()}_share_pct'] = np.round(100 * cnt / np.maximum(total, 1), 1)
    wards['land_km2'] = np.round(total * cell_km2, 2)
    ward_summary = wards.to_crs(CONFIG['web_crs'])
    name_col = next((c for c in wards.columns if 'name' in c.lower()), None)
    cols = ([name_col] if name_col else []) + ['high_share_pct', 'land_km2']
    print('Wards with the largest share of High flood risk:')
    print(ward_summary.sort_values('high_share_pct', ascending=False)[cols].head(10).to_string(index=False))
else:
    print('No ward layer in the GeoPackage: skipped.')

## 15. Save the polygons to the GeoPackage
The GeoPackage is copied from Drive to local storage, the new layers are written into it (existing layers are kept), and it is copied back.

In [ ]:
LOCAL_GPKG = f'{WORK}/bengaluru.gpkg'
shutil.copy(GPKG, LOCAL_GPKG)

flood_zones.to_file(LOCAL_GPKG, layer='flood_zones', driver='GPKG')
if ward_summary is not None:
    ward_summary.to_file(LOCAL_GPKG, layer='ward_flood_summary', driver='GPKG')

shutil.copy(LOCAL_GPKG, GPKG)
print('Layers now in the GeoPackage:', gpd.list_layers(GPKG)['name'].tolist())

## 16. Record the model in `config.json`
Saving the exact settings, weights and class breaks means Notebook 05 (validation) and the web app use the same numbers, and every change during tuning leaves a record.

In [ ]:
from datetime import datetime, timezone

CONFIG['flood_model'] = {
    **FLOOD_MODEL,
    'weights_used': WEIGHTS,
    'twi_breaks': twi_breaks,
    'class_breaks_fri': class_breaks,
    'class_area_km2': summary['area_km2'].to_dict(),
    'run_at': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC'),
}
with open(f'{PROJECT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(json.dumps(CONFIG['flood_model'], indent=2))

## 17. Summary

In [ ]:
print('Outputs in', OUT)
for f in sorted(os.listdir(OUT)):
    print(f'  {f:<28} {os.path.getsize(f"{OUT}/{f}") / 1e6:6.1f} MB')
print(f'\nHigh flood risk: {summary.loc["High", "area_km2"]} km² ({summary.loc["High", "share_%"]}% of city land)')
print('\nPhase 2 complete. Next: Notebook 03, the extreme-heat model.')